# D400 — Understanding Data Flow with Everyday Examples

The examples of Water, River, Dam, Farms used to explain to entry level interns.

## 1. Start with a river and a dam

Imagine a river flowing from the hills into a dam's reservoir. Water is then released into a canal that supplies farms.

```text
Hills and river  ──→  Dam and reservoir  ──→  Canal  ──→  Farms
```

When you stand at the dam:

- The river bringing water **towards the dam** is upstream.
- The canal carrying water **away from the dam** is downstream.

The farmers depend on water released from the dam. The dam depends on water arriving from the river.

Now imagine someone asks a farmer, “Where is your water coming from?” The farmer may point towards the dam. **The dam is upstream of the farm, even though it is downstream of the river.**

Nothing moved to a different place. We are simply talking about different parts of the same journey.

## 2. Replace water with order data

An online shop records customer orders in **MySQL**, a relational database that stores data in tables.

The data team copies those orders into **Hadoop**, an ecosystem used for distributed storage and processing. Distributed means the work or storage can be spread across multiple computers. In this example, Hadoop is the place where the team keeps incoming order data.

```text
MySQL order database  ──→  Ingestion job  ──→  Hadoop
```

**Ingestion** means bringing data into a system. A **job** is a piece of work that runs, such as copying yesterday's orders.

Here, we can simply say:

> “MySQL is our upstream system, and Hadoop is our downstream system.”

Why? Orders come **from MySQL** and go **into Hadoop**.

This is like water coming from the river and entering the reservoir. One difference: data is usually copied, so the orders can remain in MySQL after reaching Hadoop.

## 3. Continue the journey: Hadoop supplies another system

The reporting team does not want to inspect every raw order. It wants clean daily sales totals.

Suppose another job reads the orders in Hadoop, cleans them, and writes daily totals into a reporting table in Snowflake. **Snowflake** is a cloud data platform used here to store reporting data. A **dashboard** displays those totals for the business.

```text
MySQL              Hadoop                 Snowflake             Dashboard
Order records ──→ Incoming order data ──→ Daily sales totals ──→ Sales chart
```

Now Hadoop plays two parts:

- It **receives** orders from MySQL, so it is downstream of MySQL.
- It **supplies** orders to the reporting job, so it is upstream of the Snowflake reporting table.

**Hadoop is both upstream and downstream in this overall flow.** That is normal. It is like the dam: receiving water from the river and supplying water to the canal.

Snowflake also receives data from one side and supplies the dashboard on the other.

## 4. How people say this in a real team

You will hear statements such as:

> “We are waiting for the upstream MySQL extract.”

This means the next step is waiting for data from MySQL.

> “The Hadoop load is complete. The downstream reporting job can start.”

This means the data has reached Hadoop and the next job can use it.

> “If we rename this field, downstream reports may break.”

This means reports that use the field may stop working correctly.

Usually the surrounding conversation makes the meaning clear. If it does not, ask: **“Which system are we talking about?”**

The formal phrase is “relative to a reference point,” but you do not need to use that phrase in everyday conversation. “Hadoop receives from MySQL and supplies Snowflake” is often easier to understand.

## 5. Where do E and L fit into this story?

**Extract, Transform, Load (ETL)** describes what the job does:

| Step | In our example |
|---|---|
| Extract | Read order records from MySQL |
| Transform | Clean dates, remove unwanted duplicates, or calculate totals |
| Load | Write the result into the destination |

For a job moving orders from MySQL to Hadoop, a useful way to speak is:

> “E reads from the upstream MySQL system. L writes into the downstream Hadoop system.”

```text
Upstream source                                  Downstream destination
     MySQL  ──→  Extract  ──→  Transform  ──→  Load  ──→  Hadoop
```

That is a helpful convention. Just remember that **E is the reading step, and L is the writing step**. MySQL is the source; Hadoop is the destination.

The extraction code may run on a separate computer that connects to MySQL. It does not have to run inside MySQL itself.

## 6. The next job has its own E and L

Now think about the next job, which prepares reporting data:

```text
Hadoop  ──→  Read orders  ──→  Prepare daily totals  ──→  Write to Snowflake
```

For this job:

- Hadoop is the upstream source.
- Reading from Hadoop is extraction.
- Preparing totals is transformation.
- Writing to Snowflake is loading.
- Snowflake is the downstream destination.

So the destination of one job can become the source of the next job.

Some teams use **Extract, Load, Transform (ELT)**: they load data first and transform it inside the destination platform. For example, they may load raw orders into Snowflake and then build reporting tables there.

Do not worry about forcing the whole company into one E, one T, and one L. A real data journey can contain several jobs, each with its own reading, processing, and writing steps.

## 7. Source, destination, producer, and consumer

These are different ways to talk about where data comes from and who uses it.

| Word | Everyday meaning | Our example |
|---|---|---|
| Source | Where this job gets data | MySQL for the ingestion job |
| Destination or target | Where this job puts data | Hadoop for the ingestion job |
| Producer | Who creates or supplies the data | Order application creating order records |
| Consumer | Who reads or uses the data | Reporting job reading Hadoop data |
| Sink | Another name for an output destination | Storage receiving the job's output |
| System of record | The trusted authority for a particular business fact | Order system for the official order status |

When the reporting job reads Hadoop, Hadoop is its source. That does not necessarily make Hadoop the official authority for an order's status. The application database may still be the system of record.

A team can also be a producer or consumer. The data team publishes sales totals; the finance team consumes them.

## 8. Pipeline and dependency: one step waits for another

A **pipeline** is the connected set of steps that moves and prepares data. Think of the whole route from the river to the farm.

A **dependency** is something a step needs before it can work properly.

For example, a sales summary needs both orders and customer information:

```text
Order data ─────┐
               ├──→ Prepare customer sales summary ──→ Publish report
Customer data ─┘
```

If customer data has not arrived, the summary may need to wait. A successful order load alone is not enough.

An **orchestrator** is the tool that coordinates these steps: start this job, wait for that job, retry a failure, and then publish the output.

A **workflow** includes the processing steps and supporting activities such as checking results or sending notifications. You may hear **Directed Acyclic Graph (DAG)** for a diagram of task dependencies whose arrows do not loop back into a cycle.

An upstream dependency can also be a check: “Publish only after validation passes.” It does not always have to be a table supplying records.

## 9. What happens if the upstream side is late?

Imagine the reservoir has not received enough water. The canal and farms may be affected even if nothing is wrong with them.

Similarly, suppose the MySQL order extract arrives two hours late:

1. Hadoop receives the new orders late.
2. The reporting job starts late.
3. Snowflake's daily totals are updated late.
4. The dashboard continues showing older totals until its refresh completes.

The dashboard may be working perfectly but still display old data because its upstream data is late.

A helpful status update is:

> “Today's MySQL order extract has not arrived. Hadoop still has yesterday's batch, so the downstream sales report has not been refreshed.”

“Upstream problem” describes where to investigate. It is not automatically a statement that another team is at fault.

## 10. Data at rest: data has been stored

Think of water stored in a reservoir. For data, **at rest** means stored in a database, file, backup, or another storage location.

In our example:

| Place | Why it is data at rest |
|---|---|
| MySQL order table | Order records are stored there |
| Extracted file saved on a server | The file is stored while waiting for the next step |
| Hadoop storage | Ingested order data is stored there |
| Snowflake reporting table | Prepared sales data is stored there |
| Downloaded report on a laptop | A saved copy is stored on the laptop |

“At rest” does not mean old or unused. A busy order database still stores data at rest, even though new orders arrive throughout the day.

**Encryption at rest** protects the stored representation using cryptographic keys. It is like protecting the contents of a storage container. It does not decide which authorized application users have a valid reason to read the data.

## 11. Data in transit: data is being sent somewhere

Think of water flowing through a pipe between the reservoir and a farm. **Data in transit** means data being transferred between systems or endpoints.

The usual expression is **in transit**, rather than “at transit.”

Examples include:

- Order records traveling from MySQL to the ingestion worker.
- A file being uploaded into Hadoop storage.
- Prepared results being sent to Snowflake.
- Query results traveling to the dashboard.
- A report being downloaded to a laptop.

```text
Stored in MySQL  ──→  Sent over a connection  ──→  Stored in Hadoop
    At rest               In transit                 At rest
```

**Transport Layer Security (TLS)** protects network communication. **Hypertext Transfer Protocol Secure (HTTPS)** is web communication protected using TLS; **Hypertext Transfer Protocol (HTTP)** is the underlying web protocol.

A protected transfer does not automatically protect the downloaded file after it is saved. The receiving system needs its own storage and access controls.

## 12. Data in use: someone is working on it

Think of a kitchen: ingredients can be stored, delivered, or actively prepared. Data also has a working stage.

**Data in use** means data being actively processed—for example, values in a program's memory while it calculates sales totals.

Follow one order:

```text
Saved in MySQL                 → At rest
Sent to the processing job     → In transit
Read and cleaned in memory     → In use
Saved in Hadoop                → At rest
Read to calculate daily sales  → In use
Sent to the reporting system   → In transit
Saved as a report              → At rest
```

One order can have several copies in different states at the same time. The original may remain stored in MySQL while a copy is being transferred or processed elsewhere.

The river story helps explain direction, but data can be copied and branched much more freely than water.

## 13. Do not mix up direction and storage state

“Upstream” tells us where data comes from in the flow. “At rest” tells us that a copy is stored. They answer different questions.

For example:

> “We read orders from upstream MySQL and send them to downstream Hadoop.”

- MySQL contains stored data at rest.
- The connection carries data in transit.
- Hadoop receives and stores another copy at rest.
- The ingestion program may process values in use along the way.

There is no rule that upstream means stored data and downstream means moving data. Both upstream and downstream systems can store, send, and process information.

## 14. Raw, staging, and curated: unpacking and preparing deliveries

Imagine a shop receiving a delivery. Boxes arrive, staff unpack and check them, and products are arranged for customers.

Data teams use similar stages:

| Term | Plain meaning | Example |
|---|---|---|
| Raw | Close to what arrived from the source | Order records copied from MySQL |
| Staging | A working or waiting area | Files waiting to be checked and loaded |
| Curated | Prepared for an agreed purpose | Clean daily sales totals |
| Serving layer | Ready for applications or reports to use | Reporting tables used by dashboards |
| Data mart | Data organized around a business subject | Sales or finance reporting datasets |

The next sections explain a common way to organize these preparation steps: **medallion architecture**, with Bronze, Silver, and Gold layers.

In Snowflake, a **stage** can specifically mean a location for data files. That is more specific than the general phrase “staging area.”

Raw data still needs protection. “We have not cleaned it yet” does not make customer details less sensitive.

## 15. Medallion architecture: Bronze, Silver, and Gold

**Medallion architecture** organizes data into layers as it becomes more useful. Think of a shop delivery: receive the boxes, check and arrange the contents, then prepare what customers need.

| Layer | Easy way to remember | What you may find there |
|---|---|---|
| **Bronze** | Received data | Source records, incoming files or events, source name, and arrival time; duplicates or incorrect values may still exist |
| **Silver** | Cleaned and checked data | Standardized dates and types, handled duplicates, checked values, and combined order/customer details |
| **Gold** | Data prepared for a business purpose | Daily sales totals, regional reports, customer summaries, and reporting tables designed for dashboards |

Bronze helps retain what arrived for investigation and reprocessing under the retention policy. Silver often keeps detailed records. Gold often contains summaries, but can also contain detailed business-ready tables.

These are design labels, not automatic guarantees of quality or security. Each layer still needs appropriate access and privacy controls. [Medallion architecture introduction](https://docs.databricks.com/aws/en/lakehouse/medallion).


## 16. Our MySQL orders moving through the three layers

One possible arrangement for our shop is:

```text
MySQL orders
     |
     v
BRONZE: incoming order records in Hadoop
     |
     v
SILVER: cleaned order records in Hadoop
     |
     v
GOLD: daily regional sales tables in Snowflake
     |
     v
Sales dashboard
```

For example, suppose the incoming delivery contains order 101 twice, with the same source version:

- **Bronze:** Keep the received records and note which delivery they came from.
- **Silver:** Apply the agreed duplicate rule, keeping one accepted version of order 101; standardize its date and check its amount.
- **Gold:** Include that accepted order once when preparing the day's sales total for its region.

Silver receives from Bronze and supplies Gold. It is therefore **downstream of Bronze and upstream of Gold**, just like the dam receiving and releasing water.

This placement is our example, not a rule that Hadoop means Bronze or Snowflake means Gold. A team could keep all three layers in one platform using separate schemas or tables.

Also, **Bronze/Silver/Gold do not mean E/T/L**. ETL names operations; these layers describe how data is organized and prepared. A job can read Bronze, transform it, and load Silver, followed by another job that prepares Gold.


## 17. Batch and streaming: deliveries versus a steady flow

**Batch processing** handles a group of records together. Think of a truck delivering all the day's parcels at once.

**Streaming processing** handles an ongoing flow of events. Think of parcels arriving regularly on a conveyor belt. An **event** records something that happened, such as an order being placed.

- Copying yesterday's orders every morning is a batch example.
- Processing order events throughout the day is a streaming example.
- Processing small groups every minute is often called **micro-batching**.

**Full load** means loading the complete selected dataset. **Incremental load** means loading additions or changes since a previous point.

**Change Data Capture (CDC)** captures source changes for downstream use, including changes such as inserts, updates, and deletes.

A streaming system can still save events in storage. “Streaming” does not mean its data is always in transit or never at rest.

## 18. Freshness and latency: when will my order appear?

Suppose a customer places an order at 09:00. It reaches Hadoop at 09:04 and appears on the dashboard at 09:10.

**Latency** is the delay between two points. Here, order-to-Hadoop delay is four minutes, and order-to-dashboard delay is ten minutes.

**Freshness** is about how current the available data is. A dashboard that refreshed just now may still contain yesterday's records if no new input arrived.

**Event time** is when the order happened. **Ingestion time** is when it entered a particular data system. **Processing time** is when a job worked on it.

**Late-arriving data** is data that turns up after the time or batch in which it was expected. Yesterday's total might need updating if an order from yesterday arrives today.

A **Service-Level Objective (SLO)** is a measurable service target. A **Service-Level Agreement (SLA)** records agreed service commitments. A team might agree that order data should appear in reporting within a specified delay.

When someone says “the pipeline is slow,” ask which part is delayed: arrival, processing, loading, or the report refresh.

## 19. Retries and backfills: doing work again without making a mess

Imagine a delivery driver is unsure whether a parcel was accepted. Sending it again could create a duplicate delivery unless the receiver checks its identity.

Data jobs face the same issue.

| Term | Plain meaning | Example |
|---|---|---|
| Retry | Try a failed step again | Repeat a timed-out transfer |
| Backfill | Fill in or rebuild past data | Reprocess last month's corrected orders |
| Replay | Read previously available events again | Process saved order events another time |
| Idempotency | Repeating work does not create an incorrect extra effect | Loading the same order twice does not create two orders |
| Deduplication | Remove unwanted duplicates | Keep one accepted copy of an event |
| Checkpoint | Save how far processing reached | Remember the last completed source position |
| Reconciliation | Compare results with expected source information | Check loaded order counts and totals |

A retry is not automatically safe. A job may have written some records before failing.

When you backfill Hadoop or a reporting table, tell downstream users that historical figures may change. Today's processing can update yesterday's or last month's results.

## 20. Schema and contracts: agreeing on the shape of a delivery

Imagine a shop expects boxes labeled with a product code and quantity. One day, the supplier changes the labels without telling anyone. The receiving process may fail.

A **schema** describes data structure, including field names and types. **Schema drift** means that structure changes. A **breaking change** causes existing consumers to stop working correctly.

If MySQL's extract renames `customer_id` to `buyer_id`, a downstream job expecting `customer_id` may fail even though the records still exist.

A **data contract** records what producer and consumer agree to supply and expect: field structure, meaning, quality, timing, and how changes will be communicated.

The word schema has another use in platforms such as Snowflake: a named container that holds tables and views. “The source schema changed” may mean the fields changed, while “create a schema” may mean create that container.

## 21. Lineage: tracing a report back to where it started

**Data lineage** is the recorded story of where data came from and how it was prepared.

Suppose the dashboard shows an unexpected sales total. You trace it backwards:

```text
Dashboard ← Snowflake daily totals ← Preparation job ← Hadoop orders ← MySQL
```

You are investigating backwards, but the normal data flow still goes from MySQL towards the dashboard.

**Impact analysis** asks who will be affected by a change. Before renaming an order field, find the jobs, tables, and dashboards that use it.

Lineage helps answer “Where should we look?” It does not automatically explain why a number is wrong. You still need to check source records and transformation rules.

## 22. Put it all together in a team update

Suppose MySQL starts sending a new region, `CENTRAL`. Hadoop receives it successfully, but the reporting job only accepts `EAST` and `WEST`.

The result:

1. The upstream extract arrived successfully.
2. The Hadoop load completed.
3. The next transformation rejected the new region.
4. Snowflake's reporting table was not refreshed.
5. The dashboard still shows older figures.

A useful update is:

> “Orders reached Hadoop, but the reporting job stopped on the new CENTRAL region. The sales dashboard still shows yesterday's figures. We are checking the new value with the source team before rerunning the report preparation.”

This is more useful than “ETL is broken.” It tells people what succeeded, what failed, and what they will see.

The data may be encrypted during transfer and storage throughout this incident. The problem is the expected business values, not necessarily a security failure.

## 23. Quick practice

Use our example:

```text
MySQL → Hadoop → Snowflake reporting table → Dashboard
```

1. Which system supplies Hadoop?
2. Is Hadoop upstream or downstream?
3. For the MySQL-to-Hadoop job, where does E read and where does L write?
4. If MySQL data arrives late, can the dashboard show old figures even when the dashboard itself works?
5. A file is saved in Hadoop. Is that copy at rest or in transit?
6. The file is being sent to another system. What state is that transfer?
7. Can the saved original remain at rest while another copy is being sent?
8. Why might loading the same batch twice create a problem?

**Suggested answers**

1. MySQL supplies its incoming order data through the ingestion job.
2. Both: downstream of MySQL and upstream of the reporting flow.
3. E reads MySQL; L writes Hadoop.
4. Yes. It may be waiting for updated upstream results.
5. At rest.
6. In transit.
7. Yes. Data can have multiple copies.
8. It can duplicate records unless the process handles repeated input correctly.

## 24. Words to remember

| Term | Everyday meaning |
|---|---|
| Upstream | The side supplying data or something we need |
| Downstream | The side receiving or depending on our output |
| Extract | Read or obtain data |
| Transform | Prepare or change data |
| Load | Write data into a destination |
| At rest | Stored |
| In transit | Being transferred |
| In use | Being processed |

| Abbreviation | Full form |
|---|---|
| ETL | Extract, Transform, Load |
| ELT | Extract, Load, Transform |
| DAG | Directed Acyclic Graph |
| TLS | Transport Layer Security |
| HTTP | Hypertext Transfer Protocol |
| HTTPS | Hypertext Transfer Protocol Secure |
| CDC | Change Data Capture |
| SLO | Service-Level Objective |
| SLA | Service-Level Agreement |

Think of Hadoop as the dam in this example: **it receives data from MySQL and supplies data for reporting. It can be downstream on one side and upstream on the other.**